In [17]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import re

# Load the raw dataset
df = pd.read_csv("data/raw/slq_music_qld_202606.csv")

### **1. Clean creator names**

In [19]:
df["creator_clean"] = df["Creator"].fillna("Unknown").str.strip()

### **2. Extract year from Date**

In [20]:
def get_year(date):
    if pd.isna(date):
        return None

    date = str(date)

    # Do not use date ranges
    if "between" in date.lower():
        return None

    # Find a 4-digit year
    match = re.search(r"(18\d{2}|19\d{2}|20\d{2})", date)

    if match:
        return int(match.group())

    return None


df["year"] = df["Date"].apply(get_year)

### **3. Create simple themes**

In [21]:
def get_theme(subject):
    if pd.isna(subject):
        return "Other"

    subject = subject.lower()

    if "war" in subject:
        return "War"

    elif "patriotic" in subject or "national songs" in subject:
        return "Patriotic / national"

    elif "piano" in subject or "waltz" in subject or "dance" in subject:
        return "Piano / dance"

    elif "songs" in subject or "song" in subject:
        return "Songs"

    elif "queensland" in subject or "brisbane" in subject:
        return "Queensland / place"

    else:
        return "Other"


df["theme"] = df["Subjects"].apply(get_theme)

### **4. Create historical periods**

In [22]:
def get_period(year):
    if pd.isna(year):
        return "Unknown"

    if year < 1900:
        return "Before 1900"

    elif year <= 1909:
        return "1900-1909"

    elif year <= 1919:
        return "1910-1919"

    elif year <= 1929:
        return "1920-1929"

    elif year <= 1939:
        return "1930-1939"

    else:
        return "1940-1959"


df["historical_period"] = df["year"].apply(get_period)

### **5. Keep the columns we need**

In [23]:
clean_df = df[
    [
        "Title",
        "creator_clean",
        "Date",
        "year",
        "historical_period",
        "theme",
        "Subjects",
        "Publisher",
        "Format"
    ]
]
clean_df["year"] = clean_df["year"].astype("Int64")

In [24]:
print("Total records:", len(clean_df))

print("\nRecords with usable years:")
print(clean_df["year"].notna().sum())

print("\nYear range:")
print(clean_df["year"].min(), "-", clean_df["year"].max())

print("\nHistorical period counts:")
print(clean_df["historical_period"].value_counts())

Total records: 87

Records with usable years:
25

Year range:
1887 - 1953

Historical period counts:
historical_period
Unknown        62
1910-1919       7
1920-1929       5
1900-1909       5
Before 1900     3
1930-1939       3
1940-1959       2
Name: count, dtype: int64


### **6. Save cleaned data**

In [25]:
clean_df.to_csv(
    "data/clean/music_qld_clean.csv",
    index=False
)

print("Done!")
print(clean_df.head())

Done!
                                               Title  \
0  A Federal song[music] /words by George Essex E...   
1  Australia unfurl the flag![music] /G. Christia...   
2  Australia[music] /words and music by Wm. Blenheim   
3  Australian anthem[music] /words by Brunton Ste...   
4  Australian coo-ee :song /words by Robert Munro...   

                           creator_clean     Date  year historical_period  \
0             Sampson, George,1861-1949.   c1899.  1899       Before 1900   
1  Neech, G. Christian(George Christian)     1914  1914         1910-1919   
2                      Blenheim, William  [19--].  <NA>           Unknown   
3    Jefferies, Richard Thomas,1841-1920  [18--?]  <NA>           Unknown   
4                        Stein, Hendrick  [19--?]  <NA>           Unknown   

                  theme                                           Subjects  \
0  Patriotic / national  Confederation of states--Australia--Songs and ...   
1                 Songs               

In [26]:
import pandas as pd

# Load the raw dataset
df_1 = pd.read_csv("data/raw/live-music-venues.csv")

# 1. Clean venue names
df_1["venue_name"] = df_1["venue_name"].fillna("Unknown").str.strip()

# 2. Clean venue addresses
df_1["venue_address"] = df_1["venue_address"].fillna("Unknown").str.strip()

# 3. Clean space type
df_1["space_type"] = df_1["space_type"].fillna("Unknown").str.strip()

# Make the text consistent
df_1["space_type"] = df_1["space_type"].str.title()

# 4. Make latitude and longitude numeric
df_1["lat"] = pd.to_numeric(df_1["lat"], errors="coerce")
df_1["lon"] = pd.to_numeric(df_1["lon"], errors="coerce")

# 5. Keep only useful columns
clean_df_1 = df_1[
    [
        "venue_name",
        "venue_address",
        "space_type",
        "lat",
        "lon"
    ]
].copy()

# 6. Save cleaned file
clean_df_1.to_csv(
    "data/clean/venues_clean.csv",
    index=False
)

print("Done!")
print(clean_df_1.head())

Done!
            venue_name                                      venue_address  \
0     Sticky Institute          Shop 10, Campbell Arcade, Melbourne, 3000   
1         Curtin House  Unit S15, Level 7, 252 Swanston Street, Melbou...   
2         The Sub Club  Unit 20, Basement, 35 Elizabeth Street, Melbou...   
3          Trades Hall                      2 Lygon Street, Carlton, 3053   
4  Victoria University                   225 King Street, Melbourne, 3000   

  space_type        lat         lon  
0        Bar -37.817800  144.966083  
1        Bar -37.811979  144.965134  
2    Unknown -37.817392  144.964340  
3        Bar -37.806838  144.966056  
4  Multi-Use -37.814981  144.954902  


In [27]:
print("Total venues:", len(clean_df_1))

print("\nVenue types:")
print(clean_df_1["space_type"].value_counts())

print("\nMissing coordinates:")
print(clean_df_1[["lat", "lon"]].isna().sum())

Total venues: 227

Venue types:
space_type
Bar                          69
Nightclub                    30
Hotel                        30
Unknown                      23
Venue                        17
Performance Space            16
Multi-Use                    11
Theatre                       7
Amusement/Theme Venue         5
Accommodation                 5
Library                       3
Public University/College     2
Church                        2
Museum                        2
Recording                     2
Casino                        1
Community Hall                1
Gallery                       1
Name: count, dtype: int64

Missing coordinates:
lat    0
lon    0
dtype: int64


In [28]:
import pandas as pd

# Load the Excel workbook
file_path = "data/raw/cultural_and_creative_activities_202122_tables_14_to_21.xlsx"

# TABLE 14 - LIVE MUSIC ATTENDANCE BY STATE
df_2 = pd.read_excel(
    file_path,
    sheet_name="Table 14",
    header=None
)

# State names
states = [
    "NSW",
    "Victoria",
    "Queensland",
    "South Australia",
    "Western Australia",
    "Tasmania",
    "Northern Territory",
    "ACT"
]

# Live music attendance rate row
attendance_rates = df_2.iloc[29, 1:9].tolist()

# Estimated number of attendees row
attendance_estimates = df_2.iloc[14, 1:9].tolist()

# Create clean dataframe
abs_state = pd.DataFrame({
    "state": states,
    "attendance_rate": attendance_rates,
    "attendees_thousands": attendance_estimates
})

# Save file
abs_state.to_csv(
    "data/clean/abs_state_attendance.csv",
    index=False
)

print("Table 14 cleaned!")
print(abs_state)

Table 14 cleaned!
                state  attendance_rate  attendees_thousands
0                 NSW             15.6               1002.0
1            Victoria             16.1                844.5
2          Queensland             18.4                761.8
3     South Australia             19.1                277.9
4   Western Australia             22.2                478.4
5            Tasmania             23.6                109.5
6  Northern Territory             26.8                 39.3
7                 ACT             16.9                 59.0


In [29]:
# TABLE 16 - ATTENDANCE BY AGE AND GENDER
df_2 = pd.read_excel(
    file_path,
    sheet_name="Table 16",
    header=None
)

# Age groups
age_groups = [
    "15-17",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65-74",
    "75+"
]

# Male attendance rates
male_rates = df_2.iloc[63, 1:9].tolist()

# Female attendance rates
female_rates = df_2.iloc[79, 1:9].tolist()

# Overall attendance rates
persons_rates = df_2.iloc[95, 1:9].tolist()


# Create dataframe for male and female comparison
abs_age_gender = pd.DataFrame({
    "age_group": age_groups,
    "male": male_rates,
    "female": female_rates
})

# Save file
abs_age_gender.to_csv(
    "data/clean/abs_age_gender.csv",
    index=False
)


# Create dataframe for overall age attendance
abs_age = pd.DataFrame({
    "age_group": age_groups,
    "attendance_rate": persons_rates
})

# Save file
abs_age.to_csv(
    "data/clean/abs_age_attendance.csv",
    index=False
)

print("Table 16 cleaned!")

print("\nMale and female:")
print(abs_age_gender)

print("\nOverall attendance:")
print(abs_age)

Table 16 cleaned!

Male and female:
  age_group  male  female
0     15-17   8.8    15.4
1     18-24  24.6    25.7
2     25-34  19.2    20.5
3     35-44  18.2    20.0
4     45-54  19.3    21.2
5     55-64  15.4    17.1
6     65-74  11.6    13.7
7       75+   6.3     8.2

Overall attendance:
  age_group  attendance_rate
0     15-17             12.1
1     18-24             25.2
2     25-34             19.9
3     35-44             19.1
4     45-54             20.2
5     55-64             16.3
6     65-74             12.8
7       75+              7.4


In [30]:
# TABLE 21 - FREQUENCY OF LIVE MUSIC ATTENDANCE
df_2 = pd.read_excel(
    file_path,
    sheet_name="Table 21",
    header=None
)

frequency = [
    "Once",
    "Twice",
    "3 times",
    "4 times",
    "5 times",
    "6-10 times",
    "11-20 times",
    "21+ times"
]

# Proportion of live music attendees
frequency_percent = df_2.iloc[24, 1:9].tolist()

abs_frequency = pd.DataFrame({
    "frequency": frequency,
    "percentage": frequency_percent
})

# Save file
abs_frequency.to_csv(
    "data/clean/abs_frequency.csv",
    index=False
)

print("Table 21 cleaned!")
print(abs_frequency)

Table 21 cleaned!
     frequency  percentage
0         Once        42.5
1        Twice        22.4
2      3 times        11.3
3      4 times         5.9
4      5 times         4.7
5   6-10 times         8.0
6  11-20 times         2.8
7    21+ times         2.3


In [31]:
# CHART 03 - CREATOR TO THEME SANKEY

# Remove unknown creators
creator_theme = clean_df[
    clean_df["creator_clean"] != "Unknown"
].copy()

# Find the top 5 creators based on number of preserved records
top_creators = (
    creator_theme["creator_clean"]
    .value_counts()
    .head(5)
    .index
)

# Keep only the top 5 creators
creator_theme = creator_theme[
    creator_theme["creator_clean"].isin(top_creators)
]

# Count how many records connect each creator to each theme
creator_theme = (
    creator_theme
    .groupby(["creator_clean", "theme"])
    .size()
    .reset_index(name="record_count")
)

# Create shorter creator names for display
creator_theme["creator_short"] = (
    creator_theme["creator_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

# Sort by creator and number of records
creator_theme = creator_theme.sort_values(
    ["creator_clean", "record_count"],
    ascending=[True, False]
)

# Save cleaned Sankey dataset
creator_theme.to_csv(
    "data/clean/creator_theme_links.csv",
    index=False
)

print("Creator-theme Sankey data cleaned!")
print(creator_theme)

Creator-theme Sankey data cleaned!
               creator_clean                 theme  record_count creator_short
0     Brier, Percy,1885-1970         Piano / dance             3         Brier
1            Caflisch, W. A.                 Other             1      Caflisch
2            Caflisch, W. A.         Piano / dance             1      Caflisch
3            Caflisch, W. A.    Queensland / place             1      Caflisch
4   Edkins, E. M.(Edwina M.)         Piano / dance             2        Edkins
5   Edkins, E. M.(Edwina M.)                 Songs             1        Edkins
6   Edkins, E. M.(Edwina M.)                   War             1        Edkins
7   Egan-Mulry, M.,1863-1934  Patriotic / national             1    Egan-Mulry
8   Egan-Mulry, M.,1863-1934         Piano / dance             1    Egan-Mulry
9   Egan-Mulry, M.,1863-1934                   War             1    Egan-Mulry
10                Wood, Stan         Piano / dance             3          Wood
